This notebook reads the per-iteration CSV logs produced by `recovery.py` → `attack.py` (via `csv_logging.py`).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Folder containing the iter_*.csv logs
DATA_DIR = Path('/protection_results')
assert DATA_DIR.exists(), f'Not found: {DATA_DIR.resolve()}'
DATA_DIR.resolve()

In [ ]:
# 1) Read all iter_*.csv into a single dataframe called iter_df
iter_csv_paths = sorted(DATA_DIR.glob('iter_*.csv'))
if not iter_csv_paths:
    raise FileNotFoundError(f'No files matching iter_*.csv under {DATA_DIR.resolve()}')

dfs = []
for p in iter_csv_paths:
    df = pd.read_csv(p)
    df['source_file'] = p.name
    dfs.append(df)

iter_df = pd.concat(dfs, ignore_index=True)

print('Loaded files:', len(iter_csv_paths))
print('iter_df shape:', iter_df.shape)
display(iter_df.head())
print('run_type values:', sorted(iter_df['run_type'].dropna().unique().tolist()))
print('iden values (sample):', iter_df['iden'].dropna().unique()[:10])

In [ ]:
RUN_TYPES = ['base', 'bb', 'dd_double_sigm', 'bb_top1', 'bb_top5', 'bb_cross_entr']

RUN_TYPE_LABELS = {
    'base': 'white-box, Loss=cross_entropy',
    'bb': 'black-box, Loss=negative_log',
    'bb_cross_entr': 'black-box, Loss=cross_entropy',
    'bb_double_sigm': 'bb_double_sigm',
    'bb_top1': 'bb_top1',
    'bb_top5': 'bb_top5',
}

# Plot selection helpers
EXCLUDE_RUN_TYPES_PLOT_1_2 = {'base', 'bb_cross_entr'}
INCLUDE_RUN_TYPES_PLOT_3 = {'base', 'bb', 'bb_cross_entr'}

In [ ]:
# Plot 1: smoothed curve by averaging across all identity batches (iden ranges)
# Excludes: base, bb_cross_entr

smooth_df = (
    iter_df
    .assign(iden=lambda d: d['iden'].astype(str).str.strip())
    .groupby(['run_type', 'iden', 'current_iter'], as_index=False)
    .agg(acc=('acc', 'mean'))
    .groupby(['run_type', 'current_iter'], as_index=False)
    .agg(acc=('acc', 'mean'))
    .sort_values(['run_type', 'current_iter'])
)

smooth_df = smooth_df[~smooth_df['run_type'].isin(EXCLUDE_RUN_TYPES_PLOT_1_2)].copy()

plt.figure(figsize=(8, 5))
for run_type, g in smooth_df.groupby('run_type', sort=True):
    label = RUN_TYPE_LABELS.get(str(run_type), str(run_type))
    plt.plot(g['current_iter'], g['acc'], marker='o', linewidth=2, label=label)

plt.xlabel('Iterations')
plt.ylabel('''Attack accuracy (smoothed over iden batches)''')
plt.title('''Attack accuracy based on victim's output modes''')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend(title='run_type')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: smoothed curve, only for runtypes: bb_cross_entr, bb, base

smooth_df_plot3 = (
    iter_df
    .assign(iden=lambda d: d['iden'].astype(str).str.strip())
    .groupby(['run_type', 'iden', 'current_iter'], as_index=False)
    .agg(acc=('acc', 'mean'))
    .groupby(['run_type', 'current_iter'], as_index=False)
    .agg(acc=('acc', 'mean'))
    .sort_values(['run_type', 'current_iter'])
)

smooth_df_plot3 = smooth_df_plot3[smooth_df_plot3['run_type'].isin(INCLUDE_RUN_TYPES_PLOT_3)].copy()

plt.figure(figsize=(8, 5))
for run_type, g in smooth_df_plot3.groupby('run_type', sort=True):
    label = RUN_TYPE_LABELS.get(str(run_type), str(run_type))
    plt.plot(g['current_iter'], g['acc'], marker='o', linewidth=2, label=label)

plt.xlabel('Iterations')
plt.ylabel('Attack accuracy (averaged per target batch)')
plt.title('How Loss function effects attack accuracy')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend(title='run_type')
plt.tight_layout()
plt.show()

In [ ]:
# Join all summary_*.csv files and write a combined CSV

summary_csv_paths = sorted(DATA_DIR.glob('summary_*.csv'))
if not summary_csv_paths:
    raise FileNotFoundError(f'No files matching summary_*.csv under {DATA_DIR.resolve()}')

summary_dfs = []
for p in summary_csv_paths:
    df = pd.read_csv(p)
    df['source_file'] = p.name
    summary_dfs.append(df)

summary_df = pd.concat(summary_dfs, ignore_index=True)
display(summary_df)

out_path = DATA_DIR / 'joined_run_type_log_loss_summary.csv'
summary_df.to_csv(out_path, index=False)
print('Wrote:', out_path.resolve())